# Axe 3 — RFormer on Colab T4

Runs `experiments/chap5/5_3_1_rformer.py` on Colab's GPU runtime
from PyCharm's Jupyter integration.

**Prereq**: connect this notebook's Jupyter server to Google Colab in
PyCharm (Settings → Languages & Frameworks → Jupyter → Add Colab), then
choose the T4 runtime.

**Filesystem note**: cells execute on Colab's VM, not your laptop.
The repo is cloned from GitHub in the first cell so all local paths
(`data/processed/`, `results/chap5/`,
`results/reference/chap5/`) resolve against the clone.

**Optional demo**: the thesis results are trained on the CPU with
`./axe3.sh` (same machine as the linear models, so costs are comparable).
Results from this notebook are never copied into `results/reference/`.

## 1. Environment check

Verify the GPU is attached and install the two deps not preinstalled
on Colab (`iisignature`; `torch` and the rest are already there).

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

!pip install -q iisignature

## 2. Fetch the repo

Clones on first run, pulls latest on subsequent runs. All later cells
assume the working directory is the repo root.

In [ ]:
import os
from google.colab import userdata

GH_USER = 'cnammastermlballa-dotcom'
GH_REPO = 'path-signature-benchmark'

# Try Colab's secrets store first. NOTE: userdata does NOT work from
# PyCharm-Colab — it only works in the Colab web UI. In PyCharm, use
# the getpass fallback below.
token = None
try:
    token = userdata.get('GH_TOKEN')
    print("Loaded GH_TOKEN from Colab secrets.")
except userdata.SecretNotFoundError:
    print("SecretNotFoundError: no secret named 'GH_TOKEN' exists.")
except userdata.NotebookAccessError:
    print("NotebookAccessError: secret exists but not granted to this notebook.")
except Exception as e:
    print(f"Cannot fetch Colab secret ({type(e).__name__}) — "
          "expected when running from PyCharm-Colab.")

# Interactive fallback: paste the PAT once, keep it in memory only.
if not token:
    import getpass
    typed = getpass.getpass(
        "Paste GH_TOKEN (hidden input), or press Enter to try public clone: "
    ).strip()
    if typed:
        token = typed

if token:
    REPO_URL = f'https://{token}@github.com/{GH_USER}/{GH_REPO}.git'
    print("Using token-authenticated clone URL.")
else:
    REPO_URL = f'https://github.com/{GH_USER}/{GH_REPO}.git'
    print("No token — falling back to public URL.")

if not os.path.exists(GH_REPO):
    !git clone {REPO_URL}
%cd {GH_REPO}
!git pull --ff-only

## 3. Run the Axe 3 pipeline

The script reads the LASSO N* chosen by Axe 1 (`5_1_2`) for CharTraj and
NATOPS: `results/chap5/optimal_N_{chartraj,natops}_LR-L1.txt`. A fresh clone
has no `results/chap5/`, so these files are copied from
`results/reference/chap5/`, where they are added with the final CPU run of
the thesis. Until then the cell stops with a message; alternatively, run
`./axe1.sh` in this notebook first (about 50 min of CPU), and the cell uses
the files it writes in `results/chap5/`.

The script then builds the multi-view signatures, fits the LASSO control
(scikit-learn, on the CPU: about 10 min for CharTraj), trains RFormer with
5 seeds, evaluates on the held-out test set, and saves
`results/chap5/5_3_1_rformer.{csv,png}` and the per-sample predictions.

In [ ]:
import os
import shutil

REF_DIR = 'results/reference/chap5'
RUN_DIR = 'results/chap5'
N_STAR_FILES = [f'optimal_N_{ds}_LR-L1.txt' for ds in ('chartraj', 'natops')]

os.makedirs(RUN_DIR, exist_ok=True)
missing = []
for name in N_STAR_FILES:
    if os.path.exists(os.path.join(RUN_DIR, name)):
        continue                      # written by ./axe1.sh in this session
    if os.path.exists(os.path.join(REF_DIR, name)):
        shutil.copy(os.path.join(REF_DIR, name), RUN_DIR)
    else:
        missing.append(name)
if missing:
    raise FileNotFoundError(
        'Final reference N* files not available yet (missing: '
        + ', '.join(missing)
        + '): run ./axe1.sh first or wait for the reference update.'
    )

!python experiments/chap5/5_3_1_rformer.py

## 4. Inspect the results

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv('results/chap5/5_3_1_rformer.csv')
display(df)
display(Image('results/chap5/5_3_1_rformer.png'))